# Weather Logic

This notebook builds airport weather intelligence for the Command Center: METAR-style live conditions, hourly forecasts, crosswind and visibility scoring, runway operating guidance, ramp safety restrictions, operational alerts, recommendations, and backend-ready payloads.

## 1. Setup

In [ ]:
import math
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)

now = datetime(2026, 4, 27, 9, 0)

## 2. Weather Configuration

In [ ]:
AIRPORT = {
    'code': 'VIDP',
    'name': 'Demo Airport (VIDP)',
    'lat': 28.5562,
    'lon': 77.1000,
    'elevation_ft': 777,
}

RUNWAYS = pd.DataFrame([
    {'runway': '10L/28R', 'primary': '28R', 'heading': 280, 'length_m': 4430, 'ils_cat': 'CAT III', 'crosswind_limit_kt': 20, 'tailwind_limit_kt': 10},
    {'runway': '10R/28L', 'primary': '28L', 'heading': 280, 'length_m': 3810, 'ils_cat': 'CAT II', 'crosswind_limit_kt': 18, 'tailwind_limit_kt': 10},
    {'runway': '09/27', 'primary': '27', 'heading': 270, 'length_m': 2810, 'ils_cat': 'CAT I', 'crosswind_limit_kt': 15, 'tailwind_limit_kt': 8},
])

VISIBILITY_CATEGORIES = [
    {'category': 'VFR', 'min_m': 5000, 'impact': 'Normal visual operations'},
    {'category': 'CAT I', 'min_m': 1200, 'impact': 'Low visibility procedures'},
    {'category': 'CAT II', 'min_m': 550, 'impact': 'Precision approach restrictions'},
    {'category': 'CAT III', 'min_m': 200, 'impact': 'Severe low visibility operations'},
    {'category': 'Below CAT III', 'min_m': 0, 'impact': 'Suspend most arrivals and departures'},
]

WEATHER_TYPES = ['Clear', 'Haze', 'Broken Clouds', 'Light Rain', 'Thunderstorm', 'Dust', 'Fog']

## 3. Weather Helper Functions

In [ ]:
def deg_to_compass(deg):
    dirs = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE', 'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW']
    return dirs[int(round(deg / 22.5)) % 16]

def mps_to_knots(mps):
    return mps * 1.94384

def crosswind_tailwind(wind_dir, wind_kt, runway_heading):
    angle = math.radians((wind_dir - runway_heading + 180) % 360 - 180)
    crosswind = abs(wind_kt * math.sin(angle))
    headwind = wind_kt * math.cos(angle)
    tailwind = abs(min(headwind, 0))
    return round(crosswind, 1), round(tailwind, 1), round(max(headwind, 0), 1)

def visibility_category(visibility_m):
    for item in VISIBILITY_CATEGORIES:
        if visibility_m >= item['min_m']:
            return item
    return VISIBILITY_CATEGORIES[-1]

def severity_score(row):
    score = 0
    score += max(0, (5000 - row['visibility_m']) / 5000 * 30)
    score += max(0, (row['crosswind_kt'] - 8) / 16 * 25)
    score += max(0, (row['gust_kt'] - 18) / 20 * 18)
    score += 18 if row['weather_type'] in ['Thunderstorm', 'Fog'] else 0
    score += 10 if row['precip_mm_hr'] > 4 else 0
    return round(min(100, score), 1)

def risk_band(score):
    if score >= 75:
        return 'critical'
    if score >= 50:
        return 'high'
    if score >= 25:
        return 'medium'
    return 'low'

## 4. Generate Live Weather and Forecast

In [ ]:
def generate_hourly_weather(hours=24):
    rows = []
    base_wind_dir = 235
    for h in range(hours):
        ts = now + timedelta(hours=h)
        storm_wave = 1 if 9 <= h <= 14 else 0
        fog_wave = 1 if h in [20, 21, 22, 23] else 0
        weather_type = random.choices(
            WEATHER_TYPES,
            weights=[32, 14, 20, 12 + storm_wave * 10, 4 + storm_wave * 12, 6, 3 + fog_wave * 18],
            k=1,
        )[0]
        wind_mps = max(1.5, np.random.normal(6.5 + storm_wave * 2.4, 1.8))
        gust_mps = wind_mps + max(0.5, np.random.normal(2.1 + storm_wave * 2.2, 1.1))
        wind_dir = int((base_wind_dir + np.random.normal(h * 1.8, 22)) % 360)
        visibility_m = int(np.clip(np.random.normal(6200 - storm_wave * 2200 - fog_wave * 3900, 1100), 180, 10000))
        if weather_type == 'Fog':
            visibility_m = int(np.clip(np.random.normal(650, 220), 150, 1400))
        precip = round(max(0, np.random.normal(1.2 + storm_wave * 3.8, 1.6)), 1) if weather_type in ['Light Rain', 'Thunderstorm'] else 0.0
        cloud_base_ft = int(np.clip(np.random.normal(2800 - storm_wave * 1100 - fog_wave * 1600, 650), 100, 6500))
        wind_kt = round(mps_to_knots(wind_mps), 1)
        gust_kt = round(mps_to_knots(gust_mps), 1)
        crosswind_kt, tailwind_kt, headwind_kt = crosswind_tailwind(wind_dir, wind_kt, 280)
        rows.append({
            'timestamp': ts,
            'hour': ts.hour,
            'temperature_c': round(np.random.normal(29 - h * 0.12, 2.1), 1),
            'humidity_pct': int(np.clip(np.random.normal(62 + storm_wave * 14 + fog_wave * 20, 9), 25, 98)),
            'pressure_hpa': int(np.clip(np.random.normal(1011 - storm_wave * 4, 3), 996, 1024)),
            'weather_type': weather_type,
            'wind_dir_deg': wind_dir,
            'wind_compass': deg_to_compass(wind_dir),
            'wind_kt': wind_kt,
            'gust_kt': gust_kt,
            'crosswind_kt': crosswind_kt,
            'tailwind_kt': tailwind_kt,
            'headwind_kt': headwind_kt,
            'visibility_m': visibility_m,
            'cloud_base_ft': cloud_base_ft,
            'precip_mm_hr': precip,
            'lightning_nm': round(np.random.uniform(2, 18), 1) if weather_type == 'Thunderstorm' else None,
        })
    df = pd.DataFrame(rows)
    df['severity_score'] = df.apply(severity_score, axis=1)
    df['risk_band'] = df['severity_score'].apply(risk_band)
    df['visibility_category'] = df['visibility_m'].apply(lambda v: visibility_category(v)['category'])
    return df

forecast_df = generate_hourly_weather(24)
live_weather = forecast_df.iloc[0].to_dict()
forecast_df.head(8)

## 5. Runway Weather Suitability

In [ ]:
def evaluate_runways(weather):
    rows = []
    for _, rwy in RUNWAYS.iterrows():
        crosswind, tailwind, headwind = crosswind_tailwind(weather['wind_dir_deg'], weather['wind_kt'], rwy['heading'])
        vis_cat = visibility_category(weather['visibility_m'])['category']
        crosswind_ok = crosswind <= rwy['crosswind_limit_kt']
        tailwind_ok = tailwind <= rwy['tailwind_limit_kt']
        visibility_ok = not (vis_cat == 'Below CAT III')
        score = 100
        score -= max(0, crosswind - 8) * 3.1
        score -= max(0, tailwind - 2) * 4.5
        score -= 35 if not visibility_ok else 0
        score -= 18 if weather['weather_type'] == 'Thunderstorm' else 0
        status = 'open' if crosswind_ok and tailwind_ok and visibility_ok else 'restricted'
        if score < 45:
            status = 'closed'
        rows.append({
            'runway': rwy['runway'],
            'active_end': rwy['primary'],
            'ils_cat': rwy['ils_cat'],
            'crosswind_kt': crosswind,
            'tailwind_kt': tailwind,
            'headwind_kt': headwind,
            'crosswind_limit_kt': rwy['crosswind_limit_kt'],
            'tailwind_limit_kt': rwy['tailwind_limit_kt'],
            'suitability_score': round(max(0, min(100, score)), 1),
            'status': status,
        })
    return pd.DataFrame(rows).sort_values(['status', 'suitability_score'], ascending=[True, False])

runway_status_df = evaluate_runways(live_weather)
preferred_runway = runway_status_df.sort_values('suitability_score', ascending=False).iloc[0]
runway_status_df

## 6. Operational Alerts

In [ ]:
def build_weather_alerts(weather, runway_status):
    alerts = []
    vis = visibility_category(weather['visibility_m'])
    if weather['visibility_m'] < 1000:
        alerts.append({'severity': 'critical', 'type': 'visibility', 'title': 'Severe low visibility', 'message': f"Visibility {weather['visibility_m']}m. Apply {vis['category']} procedures and restrict vehicle movement."})
    elif weather['visibility_m'] < 5000:
        alerts.append({'severity': 'warning', 'type': 'visibility', 'title': 'Reduced visibility', 'message': f"Visibility {weather['visibility_m']}m. Enhanced lighting and spacing required."})

    if weather['crosswind_kt'] > 15:
        alerts.append({'severity': 'critical', 'type': 'crosswind', 'title': 'Crosswind limit pressure', 'message': f"Crosswind component {weather['crosswind_kt']} kt on active heading. Review aircraft limits."})
    elif weather['crosswind_kt'] > 8:
        alerts.append({'severity': 'warning', 'type': 'crosswind', 'title': 'Crosswind advisory', 'message': f"Crosswind component {weather['crosswind_kt']} kt. Monitor approach stability."})

    if weather['gust_kt'] > 25:
        alerts.append({'severity': 'critical', 'type': 'ramp', 'title': 'High wind ramp restriction', 'message': f"Gusts {weather['gust_kt']} kt. Secure loose equipment and restrict pushback windows."})

    if weather['weather_type'] == 'Thunderstorm' and weather['lightning_nm'] is not None and weather['lightning_nm'] <= 8:
        alerts.append({'severity': 'critical', 'type': 'lightning', 'title': 'Lightning within safety radius', 'message': f"Lightning {weather['lightning_nm']} NM from field. Stop fueling and open-ramp work."})

    restricted_runways = runway_status[runway_status['status'] != 'open']
    for _, row in restricted_runways.iterrows():
        alerts.append({'severity': 'warning', 'type': 'runway', 'title': f"{row['runway']} {row['status']}", 'message': f"Suitability {row['suitability_score']} with {row['crosswind_kt']} kt crosswind and {row['tailwind_kt']} kt tailwind."})

    if not alerts:
        alerts.append({'severity': 'ok', 'type': 'weather', 'title': 'Weather normal', 'message': 'No active weather restrictions for flight or ramp operations.'})
    return pd.DataFrame(alerts)

alerts_df = build_weather_alerts(live_weather, runway_status_df)
alerts_df

## 7. Capacity and Delay Impact Forecast

In [ ]:
def capacity_from_weather(row):
    base_capacity = 56
    capacity = base_capacity
    capacity -= max(0, row['crosswind_kt'] - 8) * 1.2
    capacity -= max(0, 5000 - row['visibility_m']) / 220
    capacity -= 9 if row['weather_type'] == 'Thunderstorm' else 0
    capacity -= 5 if row['weather_type'] == 'Fog' else 0
    capacity = int(np.clip(capacity, 12, base_capacity))
    expected_demand = int(np.clip(np.random.normal(46 + (8 <= row['hour'] <= 13) * 10 + (17 <= row['hour'] <= 20) * 8, 7), 22, 72))
    queue = max(0, expected_demand - capacity)
    delay_min = round(queue * np.random.uniform(2.1, 4.2), 1)
    return pd.Series({'arrival_departure_capacity': capacity, 'expected_movements': expected_demand, 'weather_queue': queue, 'expected_delay_min': delay_min})

capacity_df = pd.concat([forecast_df[['timestamp', 'hour', 'weather_type', 'risk_band', 'severity_score']], forecast_df.apply(capacity_from_weather, axis=1)], axis=1)
capacity_df.head(12)

## 8. AI Weather Recommendations

In [ ]:
def build_recommendations(weather, runway_status, capacity):
    recs = []
    preferred = runway_status.sort_values('suitability_score', ascending=False).iloc[0]
    recs.append({'priority': 'high' if preferred['status'] != 'open' else 'medium', 'area': 'runway', 'action': f"Prefer runway {preferred['active_end']} ({preferred['runway']})", 'reason': f"Best suitability score {preferred['suitability_score']} under current wind and visibility."})

    peak_weather = capacity.sort_values('expected_delay_min', ascending=False).iloc[0]
    if peak_weather['expected_delay_min'] > 20:
        recs.append({'priority': 'high', 'area': 'schedule', 'action': f"Pre-tactically meter movements near {peak_weather['timestamp'].strftime('%H:%M')}", 'reason': f"Weather queue forecast {peak_weather['weather_queue']} movements and {peak_weather['expected_delay_min']} min delay."})

    if weather['visibility_m'] < 5000:
        recs.append({'priority': 'high', 'area': 'ground_ops', 'action': 'Activate low-visibility ground route plan', 'reason': f"Visibility is {weather['visibility_m']}m with {visibility_category(weather['visibility_m'])['category']} operating category."})
    if weather['gust_kt'] > 22:
        recs.append({'priority': 'medium', 'area': 'ramp', 'action': 'Secure GPUs, cones, stairs, and baggage carts', 'reason': f"Wind gusts are {weather['gust_kt']} kt."})
    if weather['weather_type'] in ['Light Rain', 'Thunderstorm']:
        recs.append({'priority': 'medium', 'area': 'turnaround', 'action': 'Add wet-ramp buffer to baggage and boarding tasks', 'reason': f"Current condition is {weather['weather_type']} with precipitation {weather['precip_mm_hr']} mm/hr."})

    return pd.DataFrame(recs)

recommendations_df = build_recommendations(live_weather, runway_status_df, capacity_df)
recommendations_df

## 9. Weather KPIs

In [ ]:
weather_kpis = {
    'airport': AIRPORT['code'],
    'condition': live_weather['weather_type'],
    'temperature_c': live_weather['temperature_c'],
    'wind_kt': live_weather['wind_kt'],
    'gust_kt': live_weather['gust_kt'],
    'wind_direction': f"{live_weather['wind_dir_deg']} deg {live_weather['wind_compass']}",
    'crosswind_kt': live_weather['crosswind_kt'],
    'visibility_m': live_weather['visibility_m'],
    'visibility_category': live_weather['visibility_category'],
    'risk_score': live_weather['severity_score'],
    'risk_band': live_weather['risk_band'],
    'open_runways': int((runway_status_df['status'] == 'open').sum()),
    'forecast_delay_min_next_24h': round(capacity_df['expected_delay_min'].sum(), 1),
    'worst_forecast_hour': capacity_df.sort_values('severity_score', ascending=False).iloc[0]['timestamp'].strftime('%H:%M'),
    'active_alerts': int((alerts_df['severity'] != 'ok').sum()),
}
weather_kpis

## 10. Weather Visuals

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(forecast_df['timestamp'], forecast_df['wind_kt'], marker='o', label='Wind')
axes[0, 0].plot(forecast_df['timestamp'], forecast_df['gust_kt'], marker='o', label='Gust')
axes[0, 0].axhline(25, color='red', linestyle='--', linewidth=1, label='High wind threshold')
axes[0, 0].set_title('Wind and Gust Forecast')
axes[0, 0].set_ylabel('Knots')
axes[0, 0].legend()
axes[0, 0].tick_params(axis='x', rotation=45)

axes[0, 1].bar(forecast_df['timestamp'], forecast_df['visibility_m'], color='#60a5fa')
axes[0, 1].axhline(5000, color='green', linestyle='--', linewidth=1, label='Normal threshold')
axes[0, 1].axhline(1000, color='orange', linestyle='--', linewidth=1, label='Severe threshold')
axes[0, 1].set_title('Visibility Forecast')
axes[0, 1].set_ylabel('Meters')
axes[0, 1].legend()
axes[0, 1].tick_params(axis='x', rotation=45)

axes[1, 0].plot(capacity_df['timestamp'], capacity_df['arrival_departure_capacity'], marker='o', label='Weather capacity')
axes[1, 0].plot(capacity_df['timestamp'], capacity_df['expected_movements'], marker='x', label='Expected demand')
axes[1, 0].set_title('Capacity vs Demand')
axes[1, 0].set_ylabel('Movements/hour')
axes[1, 0].legend()
axes[1, 0].tick_params(axis='x', rotation=45)

axes[1, 1].bar(runway_status_df['active_end'], runway_status_df['suitability_score'], color=['#34d399' if s == 'open' else '#fbbf24' if s == 'restricted' else '#ef4444' for s in runway_status_df['status']])
axes[1, 1].set_ylim(0, 100)
axes[1, 1].set_title('Runway Suitability')
axes[1, 1].set_ylabel('Score')

plt.tight_layout()
plt.show()

## 11. Backend Payload

In [ ]:
def build_weather_payload():
    return {
        'generated_at': now.isoformat(),
        'airport': AIRPORT,
        'kpis': weather_kpis,
        'current': {k: (v.item() if hasattr(v, 'item') else v) for k, v in live_weather.items()},
        'runways': runway_status_df.to_dict(orient='records'),
        'alerts': alerts_df.to_dict(orient='records'),
        'forecast': forecast_df.head(24).assign(timestamp=lambda d: d['timestamp'].astype(str)).to_dict(orient='records'),
        'capacity_forecast': capacity_df.assign(timestamp=lambda d: d['timestamp'].astype(str)).to_dict(orient='records'),
        'recommendations': recommendations_df.to_dict(orient='records'),
    }

payload = build_weather_payload()
payload.keys()

## 12. Summary

In [ ]:
print('WEATHER SUMMARY')
print('===============')
print(f"Airport: {AIRPORT['name']}")
print(f"Condition: {weather_kpis['condition']}")
print(f"Wind: {weather_kpis['wind_kt']} kt gusting {weather_kpis['gust_kt']} kt from {weather_kpis['wind_direction']}")
print(f"Crosswind: {weather_kpis['crosswind_kt']} kt")
print(f"Visibility: {weather_kpis['visibility_m']} m ({weather_kpis['visibility_category']})")
print(f"Risk: {weather_kpis['risk_score']} ({weather_kpis['risk_band']})")
print(f"Open runways: {weather_kpis['open_runways']}")
print(f"Forecast delay next 24h: {weather_kpis['forecast_delay_min_next_24h']} min")

print('\nTop weather recommendations:')
for _, rec in recommendations_df.head(5).iterrows():
    print(f"- [{rec['priority']}] {rec['action']} - {rec['reason']}")